In [25]:
print("Zepto Data Pipeline - Module 1")
print("Environment is working!")

Zepto Data Pipeline - Module 1
Environment is working!


In [26]:
%pip install requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import sqlite3

print("requests:", requests.__version__)
print("pandas:", pd.__version__)
print("BeautifulSoup: OK")
print("sqlite3: OK")

requests: 2.34.2
pandas: 3.0.5
BeautifulSoup: OK
sqlite3: OK


In [28]:
import os

DATA_DIR = "data"
OUTPUT_DIR = "outputs"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project directories created:")
print(f"- {DATA_DIR}/")
print(f"- {OUTPUT_DIR}/")

Project directories created:
- data/
- outputs/


In [29]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders:")
print(os.listdir("."))

Current working directory:
c:\Users\ABC\Desktop\capstone project\module 1\data_pipeline

Files and folders:
['data', 'data_pipeline.ipynb', 'outputs']


## 1. Project Configuration

In [30]:
import os

# Project directories
DATA_DIR = "data"
OUTPUT_DIR = "outputs"

# SQLite database path
DB_PATH = os.path.join(DATA_DIR, "books.db")

# Fixed currency conversion rate required by the assignment
GBP_TO_INR = 105.50

# Website
BASE_URL = "https://books.toscrape.com/"

print("Configuration loaded successfully.")
print("Base URL:", BASE_URL)
print("GBP → INR:", GBP_TO_INR)
print("Database:", DB_PATH)

Configuration loaded successfully.
Base URL: https://books.toscrape.com/
GBP → INR: 105.5
Database: data\books.db


In [31]:
# ==========================================
# Final Environment Validation
# ==========================================

import requests
import pandas as pd
import sqlite3
from bs4 import BeautifulSoup

# 1. Test website connectivity
response = requests.get(BASE_URL, timeout=15)

print("Website status code:", response.status_code)

# 2. Test BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")
print("BeautifulSoup: OK")

# 3. Test pandas
test_df = pd.DataFrame({
    "test": [1, 2, 3]
})
print("Pandas: OK")

# 4. Test SQLite
connection = sqlite3.connect(":memory:")
connection.execute("SELECT 1")
connection.close()
print("SQLite: OK")

# 5. Final status
if response.status_code == 200:
    print("\n✅ Stage 1 Environment Validation PASSED")
else:
    print("\n❌ Website connection failed")

Website status code: 200
BeautifulSoup: OK
Pandas: OK
SQLite: OK

✅ Stage 1 Environment Validation PASSED


## 2. Data Extraction — Web Scraping

### Objective

Scrape book catalog data from Books to Scrape using the `requests` and `BeautifulSoup` libraries.

For each book, collect:

- Title
- Price in GBP
- Star rating as text
- Availability text
- Category

The final dataset must contain at least 60 books across at least 3 categories.

In [32]:
# ==========================================
# Categories to Scrape
# ==========================================

CATEGORIES = {
    "Travel": "catalogue/category/books/travel_2/index.html",
    "Mystery": "catalogue/category/books/mystery_3/index.html",
    "Science": "catalogue/category/books/science_22/index.html",
    "Classics": "catalogue/category/books/classics_6/index.html",
    "Historical Fiction": "catalogue/category/books/historical-fiction_4/index.html",
    "Business": "catalogue/category/books/business_35/index.html",
    "Thriller": "catalogue/category/books/thriller_37/index.html"
}

print("Categories selected:")
for category in CATEGORIES:
    print("-", category)

Categories selected:
- Travel
- Mystery
- Science
- Classics
- Historical Fiction
- Business
- Thriller


In [33]:
# HTTP request headers
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )
}

print("HEADERS configured successfully.")

HEADERS configured successfully.


In [34]:
# ==========================================
# Scraping Function
# ==========================================
from urllib.parse import urljoin
def scrape_category(category_name, category_url):

    books = []

    # Convert relative URL to absolute URL
    next_page = urljoin(BASE_URL, category_url)

    while next_page:

        print(f"Scraping {category_name}: {next_page}")

        try:
            response = requests.get(
                next_page,
                headers=HEADERS,
                timeout=15
            )

            response.raise_for_status()

        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            break

        # Parse HTML
        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # Find all book cards
        articles = soup.find_all(
            "article",
            class_="product_pod"
        )

        for article in articles:

            try:
                # Book title
                title = article.h3.a["title"]

                # Price
                price = article.find(
                    "p",
                    class_="price_color"
                ).get_text(strip=True)

                # Star rating
                rating = article.find(
                    "p",
                    class_="star-rating"
                )["class"][1]

                # Availability
                availability = article.find(
                    "p",
                    class_="instock availability"
                ).get_text(" ", strip=True)

                # Store book
                books.append({
                    "title": title,
                    "price": price,
                    "star_rating": rating,
                    "availability": availability,
                    "category": category_name
                })

            except (AttributeError, KeyError, TypeError) as e:
                print(f"Skipping malformed book: {e}")
                continue

        # Check for next page
        next_button = soup.find(
            "li",
            class_="next"
        )

        if next_button:

            href = next_button.find("a")["href"]

            next_page = urljoin(
                next_page,
                href
            )

        else:
            next_page = None

    return books

In [35]:
# ==========================================
# Execute Scraping
# ==========================================

all_books = []

for category_name, category_url in CATEGORIES.items():

    category_books = scrape_category(
        category_name,
        category_url
    )

    all_books.extend(category_books)

print("\nScraping completed.")
print("Total books scraped:", len(all_books))

Scraping Travel: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Scraping Mystery: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scraping Mystery: https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
Scraping Science: https://books.toscrape.com/catalogue/category/books/science_22/index.html
Scraping Classics: https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Scraping Historical Fiction: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scraping Historical Fiction: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
Scraping Business: https://books.toscrape.com/catalogue/category/books/business_35/index.html
Scraping Thriller: https://books.toscrape.com/catalogue/category/books/thriller_37/index.html

Scraping completed.
Total books scraped: 125


In [39]:
# ==========================================
# Create Raw DataFrame
# ==========================================

raw_df = pd.DataFrame(all_books)

print("Dataset shape:", raw_df.shape)

raw_df.head(10)

Dataset shape: (125, 5)


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
5,A Summer In Europe,Â£44.34,Two,In stock,Travel
6,The Great Railway Bazaar,Â£30.54,One,In stock,Travel
7,A Year in Provence (Provence #1),Â£56.88,Four,In stock,Travel
8,The Road to Little Dribbling: Adventures of an...,Â£23.21,One,In stock,Travel
9,Neither Here nor There: Travels in Europe,Â£38.95,Three,In stock,Travel


In [41]:
# ==========================================
# Validate Required Columns
# ==========================================

required_columns = [
    "title",
    "price",
    "star_rating",
    "availability",
    "category"
]

missing_columns = [
    column
    for column in required_columns
    if column not in raw_df.columns
]

if not missing_columns:
    print("✅ All required raw columns are present.")
else:
    print("❌ Missing columns:", missing_columns)

✅ All required raw columns are present.


In [42]:
# ==========================================
# Validate Scraping Requirements
# ==========================================

total_books = len(raw_df)
total_categories = raw_df["category"].nunique()

print("Total books scraped:", total_books)
print("Total categories:", total_categories)

print("\nBooks per category:")
print(raw_df["category"].value_counts())

assert total_books >= 60, "❌ Fewer than 60 books scraped."
assert total_categories >= 3, "❌ Fewer than 3 categories scraped."

print("\n✅ Minimum scraping requirements satisfied.")

Total books scraped: 125
Total categories: 7

Books per category:
category
Mystery               32
Historical Fiction    26
Classics              19
Science               14
Business              12
Travel                11
Thriller              11
Name: count, dtype: int64

✅ Minimum scraping requirements satisfied.


In [43]:
# ==========================================
# Raw Data Quality Check
# ==========================================

print("Missing values:")
print(raw_df.isnull().sum())

print("\nDuplicate rows:")
print(raw_df.duplicated().sum())

Missing values:
title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64

Duplicate rows:
0


In [44]:
raw_df.head(10)

,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
5,A Summer In Europe,Â£44.34,Two,In stock,Travel
6,The Great Railway Bazaar,Â£30.54,One,In stock,Travel
7,A Year in Provence (Provence #1),Â£56.88,Four,In stock,Travel
8,The Road to Little Dribbling: Adventures of an...,Â£23.21,One,In stock,Travel
9,Neither Here nor There: Travels in Europe,Â£38.95,Three,In stock,Travel


In [45]:
raw_df.tail(5)

,title,price,star_rating,availability,category
120,Give It Back,Â£18.32,Two,In stock,Thriller
121,Killing Floor (Jack Reacher #1),Â£31.49,Four,In stock,Thriller
122,The Bone Hunters (Lexy Vaughan & Steven Macaul...,Â£59.71,Three,In stock,Thriller
123,Far From True (Promise Falls Trilogy #2),Â£34.93,Two,In stock,Thriller
124,The Travelers,Â£15.77,One,In stock,Thriller


In [46]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   title         125 non-null    str  
 1   price         125 non-null    str  
 2   star_rating   125 non-null    str  
 3   availability  125 non-null    str  
 4   category      125 non-null    str  
dtypes: str(5)
memory usage: 5.0 KB


In [47]:
# ==========================================
# Save Raw Dataset
# ==========================================

raw_file = os.path.join(
    DATA_DIR,
    "raw_books.csv"
)

raw_df.to_csv(
    raw_file,
    index=False,
    encoding="utf-8"
)

print(f"✅ Raw dataset saved to: {raw_file}")

✅ Raw dataset saved to: data\raw_books.csv


In [48]:
import os

print("Raw CSV exists:", os.path.exists(raw_file))
print("Raw CSV path:", raw_file)

Raw CSV exists: True
Raw CSV path: data\raw_books.csv


## 3. Data Cleaning & Transformation

### Objective

Transform the raw scraped fields into analysis-ready types.

Cleaning operations:

1. Convert price from GBP text to `price_gbp`.
2. Convert textual star ratings to integers from 1–5.
3. Convert availability text to the boolean `in_stock`.
4. Calculate `price_inr` using the required fixed rate of **1 GBP = 105.50 INR**.
5. Handle unexpected parsing failures without crashing the pipeline.

In [49]:
# ==========================================
# Create Working Copy
# ==========================================

cleaned_df = raw_df.copy()

print("Working copy created.")
print("Rows:", len(cleaned_df))

Working copy created.
Rows: 125


In [51]:
# ==========================================
# Clean Price
# ==========================================

cleaned_df["price_gbp"] = (
    cleaned_df["price"]
    .astype("string")
    .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
)

cleaned_df["price_gbp"] = pd.to_numeric(
    cleaned_df["price_gbp"],
    errors="coerce"
)

print(cleaned_df[["price", "price_gbp"]].head())

     price  price_gbp
0  Â£45.17      45.17
1  Â£49.43      49.43
2  Â£48.87      48.87
3  Â£36.94      36.94
4  Â£37.33      37.33


In [52]:
print("Missing price values:", cleaned_df["price_gbp"].isna().sum())
print("Price data type:", cleaned_df["price_gbp"].dtype)

Missing price values: 0
Price data type: Float64


In [67]:
# ==========================================
# Convert Star Rating
# ==========================================

RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

cleaned_df["rating"] = cleaned_df["star_rating"].map(RATING_MAP)

print(
    cleaned_df[
        ["star_rating", "rating"]
    ].head(10)
)

KeyError: 'star_rating'

In [68]:
# ==========================================
# Convert Availability to Boolean
# ==========================================

cleaned_df["in_stock"] = (
    cleaned_df["availability"]
    .astype("string")
    .str.contains(
        "In stock",
        case=False,
        na=False
    )
)

print(
    cleaned_df[
        ["availability", "in_stock"]
    ].head(10)
)

KeyError: 'availability'

In [69]:
# ==========================================
# Handle Numeric Parsing Failures
# ==========================================

price_missing_before = cleaned_df["price_gbp"].isna().sum()
rating_missing_before = cleaned_df["rating"].isna().sum()

print("Invalid price values:", price_missing_before)
print("Invalid rating values:", rating_missing_before)


# Median imputation for invalid price values
if cleaned_df["price_gbp"].isna().any():
    price_median = cleaned_df["price_gbp"].median()

    cleaned_df["price_gbp"] = cleaned_df["price_gbp"].fillna(
        price_median
    )


# Median imputation for invalid rating values
if cleaned_df["rating"].isna().any():
    rating_median = cleaned_df["rating"].median()

    cleaned_df["rating"] = cleaned_df["rating"].fillna(
        rating_median
    )


print("\nNumeric parsing failures handled.")

Invalid price values: 0
Invalid rating values: 0

Numeric parsing failures handled.


In [70]:
# ==========================================
# GBP → INR Conversion
# ==========================================

cleaned_df["price_inr"] = (
    cleaned_df["price_gbp"] * GBP_TO_INR
).round(2)

print(
    cleaned_df[
        ["price_gbp", "price_inr"]
    ].head(10)
)

   price_gbp  price_inr
0      45.17    4765.44
1      49.43    5214.86
2      48.87    5155.78
3      36.94    3897.17
4      37.33    3938.31
5      44.34    4677.87
6      30.54    3221.97
7      56.88    6000.84
8      23.21    2448.66
9      38.95    4109.23


In [71]:
# ==========================================
# Handle Missing Essential Fields
# ==========================================

rows_before = len(cleaned_df)

cleaned_df = cleaned_df.dropna(
    subset=["title", "category"]
).copy()

rows_after = len(cleaned_df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows dropped:", rows_before - rows_after)

Rows before: 125
Rows after: 125
Rows dropped: 0


In [72]:
# ==========================================
# Final Cleaned Dataset
# ==========================================

cleaned_df = cleaned_df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].copy()

cleaned_df.head(10)

,title,price_gbp,price_inr,rating,in_stock,category
0,It's Only the Himalayas,45.17,4765.44,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.78,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,True,Travel
4,Under the Tuscan Sun,37.33,3938.31,3,True,Travel
5,A Summer In Europe,44.34,4677.87,2,True,Travel
6,The Great Railway Bazaar,30.54,3221.97,1,True,Travel
7,A Year in Provence (Provence #1),56.88,6000.84,4,True,Travel
8,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1,True,Travel
9,Neither Here nor There: Travels in Europe,38.95,4109.23,3,True,Travel


In [73]:
# ==========================================
# Enforce Required Data Types
# ==========================================

cleaned_df["title"] = cleaned_df["title"].astype("string")
cleaned_df["price_gbp"] = cleaned_df["price_gbp"].astype(float)
cleaned_df["price_inr"] = cleaned_df["price_inr"].astype(float)
cleaned_df["rating"] = cleaned_df["rating"].astype(int)
cleaned_df["in_stock"] = cleaned_df["in_stock"].astype(bool)
cleaned_df["category"] = cleaned_df["category"].astype("string")

print(cleaned_df.dtypes)

title         string
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      string
dtype: object


In [74]:
# ==========================================
# Final Cleaning Validation
# ==========================================

print("Dataset shape:", cleaned_df.shape)

print("\nData types:")
print(cleaned_df.dtypes)

print("\nMissing values:")
print(cleaned_df.isnull().sum())

print("\nRating values:")
print(sorted(cleaned_df["rating"].unique()))

print("\nStock values:")
print(cleaned_df["in_stock"].unique())

Dataset shape: (125, 6)

Data types:
title         string
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      string
dtype: object

Missing values:
title        0
price_gbp    0
price_inr    0
rating       0
in_stock     0
category     0
dtype: int64

Rating values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Stock values:
[ True]


In [75]:
# ==========================================
# Verify GBP → INR Calculation
# ==========================================

calculated_price_inr = (
    cleaned_df["price_gbp"] * GBP_TO_INR
).round(2)

conversion_check = (
    cleaned_df["price_inr"] == calculated_price_inr
)

print("All currency conversions correct:",
      conversion_check.all())

All currency conversions correct: True


In [76]:
# ==========================================
# Save Cleaned Dataset
# ==========================================

cleaned_file = os.path.join(
    DATA_DIR,
    "cleaned_books.csv"
)

cleaned_df.to_csv(
    cleaned_file,
    index=False,
    encoding="utf-8"
)

print(f"✅ Cleaned dataset saved to: {cleaned_file}")

✅ Cleaned dataset saved to: data\cleaned_books.csv


## 4. SQLite Database — Load

### Objective

Load the cleaned catalog data into a normalized SQLite relational database.

The database contains two related tables:

- `categories` — stores unique book categories.
- `books` — stores book-level information and references `categories` through a foreign key.

Relationship:

`categories.category_id → books.category_id`

In [77]:
# ==========================================
# SQLite Setup
# ==========================================

import sqlite3

print("SQLite module imported successfully.")
print("Database path:", DB_PATH)

SQLite module imported successfully.
Database path: data\books.db


In [78]:
# ==========================================
# Create SQLite Connection
# ==========================================

conn = sqlite3.connect(DB_PATH)

print("Connected to SQLite database successfully.")

Connected to SQLite database successfully.


In [79]:
# Enable foreign key enforcement
conn.execute("PRAGMA foreign_keys = ON;")

foreign_keys_status = conn.execute(
    "PRAGMA foreign_keys;"
).fetchone()[0]

print("Foreign key enforcement:", foreign_keys_status)

Foreign key enforcement: 1


In [80]:
# ==========================================
# Create Normalized Database Schema
# ==========================================

cursor = conn.cursor()

# Remove old tables if they exist
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

# Categories table
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT NOT NULL UNIQUE
)
""")

# Books table
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("Database schema created successfully.")

Database schema created successfully.


In [81]:
# ==========================================
# Verify Database Tables
# ==========================================

tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", conn)

tables

,name
0,books
1,categories
2,sqlite_sequence


In [82]:
# ==========================================
# Insert Categories
# ==========================================

unique_categories = (
    cleaned_df["category"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

for category in unique_categories:
    cursor.execute(
        """
        INSERT INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

conn.commit()

print("Categories inserted:", len(unique_categories))

Categories inserted: 7


In [84]:
# ==========================================
# Create Category ID Lookup
# ==========================================

category_lookup_df = pd.read_sql("""
SELECT category_id, category_name
FROM categories
ORDER BY category_id
""", conn)

category_lookup_df

,category_id,category_name
0,1,Travel
1,2,Mystery
2,3,Science
3,4,Classics
4,5,Historical Fiction
5,6,Business
6,7,Thriller


In [85]:
category_lookup = dict(
    zip(
        category_lookup_df["category_name"],
        category_lookup_df["category_id"]
    )
)

print(category_lookup)

{'Travel': 1, 'Mystery': 2, 'Science': 3, 'Classics': 4, 'Historical Fiction': 5, 'Business': 6, 'Thriller': 7}


In [86]:
# ==========================================
# Load Books into SQLite
# ==========================================

books_to_insert = cleaned_df.copy()

# Map category names to category IDs
books_to_insert["category_id"] = (
    books_to_insert["category"]
    .map(category_lookup)
)

# Verify that every category was mapped
missing_category_ids = books_to_insert["category_id"].isna().sum()

print("Books with missing category IDs:", missing_category_ids)

Books with missing category IDs: 0


In [ ]:
# ==========================================
# Insert Books
# ==========================================

for _, row in books_to_insert.iterrows():

    cursor.execute(
        """
        INSERT INTO books (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            float(row["price_gbp"]),
            float(row["price_inr"]),
            int(row["rating"]),
            int(row["in_stock"]),
            int(row["category_id"])
        )
    )

conn.commit()

print("Books inserted successfully.")

Books inserted successfully.


In [88]:
# ==========================================
# Verify Database Row Counts
# ==========================================

category_count = pd.read_sql(
    "SELECT COUNT(*) AS total_categories FROM categories",
    conn
)

book_count = pd.read_sql(
    "SELECT COUNT(*) AS total_books FROM books",
    conn
)

print(category_count)
print(book_count)

   total_categories
0                 7
   total_books
0          125


In [89]:
# ==========================================
# Foreign Key Integrity Check
# ==========================================

foreign_key_check = pd.read_sql(
    """
    SELECT *
    FROM books
    WHERE category_id NOT IN (
        SELECT category_id
        FROM categories
    )
    """,
    conn
)

print("Books with invalid category references:")
print(len(foreign_key_check))

Books with invalid category references:
0


In [90]:
books_preview = pd.read_sql(
    """
    SELECT *
    FROM books
    LIMIT 10
    """,
    conn
)

books_preview

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.44,2,1,1
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1,1
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,1,1
4,5,Under the Tuscan Sun,37.33,3938.31,3,1,1
5,6,A Summer In Europe,44.34,4677.87,2,1,1
6,7,The Great Railway Bazaar,30.54,3221.97,1,1,1
7,8,A Year in Provence (Provence #1),56.88,6000.84,4,1,1
8,9,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1,1,1
9,10,Neither Here nor There: Travels in Europe,38.95,4109.23,3,1,1


In [91]:
relationship_preview = pd.read_sql(
    """
    SELECT
        b.book_id,
        b.title,
        b.price_gbp,
        b.rating,
        b.in_stock,
        c.category_name
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
    LIMIT 10
    """,
    conn
)

relationship_preview

,book_id,title,price_gbp,rating,in_stock,category_name
0,1,It's Only the Himalayas,45.17,2,1,Travel
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,4,1,Travel
2,3,See America: A Celebration of Our National Par...,48.87,3,1,Travel
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,2,1,Travel
4,5,Under the Tuscan Sun,37.33,3,1,Travel
5,6,A Summer In Europe,44.34,2,1,Travel
6,7,The Great Railway Bazaar,30.54,1,1,Travel
7,8,A Year in Provence (Provence #1),56.88,4,1,Travel
8,9,The Road to Little Dribbling: Adventures of an...,23.21,1,1,Travel
9,10,Neither Here nor There: Travels in Europe,38.95,3,1,Travel


In [92]:
# ==========================================
# Final Database Validation
# ==========================================

expected_books = len(cleaned_df)
actual_books = pd.read_sql(
    "SELECT COUNT(*) AS count FROM books",
    conn
)["count"].iloc[0]

expected_categories = cleaned_df["category"].nunique()
actual_categories = pd.read_sql(
    "SELECT COUNT(*) AS count FROM categories",
    conn
)["count"].iloc[0]

print("Expected books:", expected_books)
print("Database books:", actual_books)

print("\nExpected categories:", expected_categories)
print("Database categories:", actual_categories)

assert actual_books == expected_books, "Book count mismatch."
assert actual_categories == expected_categories, "Category count mismatch."

print("\n✅ Database validation PASSED")

Expected books: 125
Database books: 125

Expected categories: 7
Database categories: 7

✅ Database validation PASSED


## 5. SQL Analysis

The SQLite database is queried to demonstrate catalog-level pricing, availability, rating, and category analysis.

The queries collectively demonstrate:

- SELECT
- WHERE
- ORDER BY
- LIMIT
- DISTINCT
- BETWEEN
- IN
- JOIN

### Query 1 — Books Priced Above £40

Retrieve books with a GBP price greater than £40.

In [94]:
# ==========================================
# SQL Query 1 — SELECT + WHERE
# ==========================================

query1 = """
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE price_gbp > 40
"""

query1_result = pd.read_sql(query1, conn)

print("Query 1 Results:")
query1_result

Query 1 Results:


,book_id,title,price_gbp,price_inr,rating,in_stock
0,1,It's Only the Himalayas,45.17,4765.44,2,1
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1
3,6,A Summer In Europe,44.34,4677.87,2,1
4,8,A Year in Provence (Provence #1),56.88,6000.84,4,1
5,12,Sharp Objects,47.82,5045.01,4,1
6,14,The Past Never Ends,56.50,5960.75,4,1
7,16,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.55,4,1
8,17,The Last Mile (Amos Decker #2),54.21,5719.16,2,1
9,20,A Time of Torment (Charlie Parker #14),48.35,5100.92,5,1


In [95]:
# Save Query 1 output
query1_output = os.path.join(
    OUTPUT_DIR,
    "query1_price_above_40.csv"
)

query1_result.to_csv(
    query1_output,
    index=False
)

print(f"Query 1 output saved to: {query1_output}")

Query 1 output saved to: outputs\query1_price_above_40.csv


In [96]:
print("Number of books returned:", len(query1_result))

assert (
    query1_result["price_gbp"] > 40
).all()

print("✅ Query 1 validation passed.")

Number of books returned: 41
✅ Query 1 validation passed.


### Query 2 — Top 10 Most Expensive Books

Retrieve the 10 books with the highest GBP prices.

In [101]:
# ==========================================
# SQL Query 2 — ORDER BY + LIMIT
# ==========================================

query2 = """
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

query2_result = pd.read_sql(query2, conn)

print("Query 2 Results:")
query2_result

Query 2 Results:


,book_id,title,price_gbp,price_inr,rating,in_stock
0,123,The Bone Hunters (Lexy Vaughan & Steven Macaul...,59.71,6299.40,3,1
1,26,Boar Island (Anna Pigeon #19),59.48,6275.14,3,1
2,64,Candide,58.63,6185.46,3,1
3,39,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35,4,1
4,45,Immunity: How Elie Metchnikoff Changed the Cou...,57.36,6051.48,5,1
5,54,The Disappearing Spoon: And Other True Tales o...,57.35,6050.42,5,1
6,65,Animal Farm,57.22,6036.71,3,1
7,8,A Year in Provence (Provence #1),56.88,6000.84,4,1
8,14,The Past Never Ends,56.50,5960.75,4,1
9,48,"The Fabric of the Cosmos: Space, Time, and the...",55.91,5898.50,1,1


In [102]:
# ==========================================
# Validate Required Columns
# ==========================================

required_columns = [
    "title",
    "price",
    "star_rating",
    "availability",
    "category"
]

missing_columns = [
    column
    for column in required_columns
    if column not in raw_df.columns
]

if not missing_columns:
    print("✅ All required raw columns are present.")
else:
    print("❌ Missing columns:", missing_columns)

✅ All required raw columns are present.


In [103]:
# Save Query 2 output

query2_output = os.path.join(
    OUTPUT_DIR,
    "query2_top10_expensive.csv"
)

query2_result.to_csv(
    query2_output,
    index=False
)

print(f"Query 2 output saved to: {query2_output}")

Query 2 output saved to: outputs\query2_top10_expensive.csv


In [104]:
print("Number of books returned:", len(query2_result))

assert len(query2_result) == 10, (
    "Query 2 should return exactly 10 books."
)

assert query2_result["price_gbp"].is_monotonic_decreasing, (
    "Books are not ordered by descending price."
)

print("✅ Query 2 validation passed.")

Number of books returned: 10
✅ Query 2 validation passed.


### Query 3 — Distinct Book Categories

Retrieve the unique categories represented in the book catalogue.

In [105]:
# ==========================================
# SQL Query 3 — DISTINCT
# ==========================================

query3 = """
SELECT DISTINCT
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY c.category_name
"""

query3_result = pd.read_sql(query3, conn)

print("Query 3 Results:")
query3_result

Query 3 Results:


,category_name
0,Business
1,Classics
2,Historical Fiction
3,Mystery
4,Science
5,Thriller
6,Travel


In [106]:
# Save Query 3 output

query3_output = os.path.join(
    OUTPUT_DIR,
    "query3_categories.csv"
)

query3_result.to_csv(
    query3_output,
    index=False
)

print(f"Query 3 output saved to: {query3_output}")

Query 3 output saved to: outputs\query3_categories.csv


In [107]:
# Validate Query 3

print("Number of unique categories:", len(query3_result))

assert (
    len(query3_result)
    == cleaned_df["category"].nunique()
), "Category count mismatch."

assert (
    query3_result["category_name"].is_unique
), "DISTINCT query returned duplicate categories."

print("✅ Query 3 validation passed.")

Number of unique categories: 7
✅ Query 3 validation passed.


### Query 4 — Books Within a GBP Price Range

Retrieve books priced between £20 and £30, inclusive.

In [111]:
# ==========================================
# SQL Query 4 — BETWEEN
# ==========================================

query4 = """
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE price_gbp BETWEEN 20 AND 30
ORDER BY price_gbp
"""

query4_result = pd.read_sql(query4, conn)

print("Query 4 Results:")
query4_result

Query 4 Results:


,book_id,title,price_gbp,price_inr,rating,in_stock
0,42,Blood Defense (Samantha Brinkman #1),20.30,2141.65,3,1
1,84,"Love, Lies and Spies",20.55,2168.02,2,1
2,97,Between Shades of Gray,20.79,2193.34,5,1
3,31,Delivering the Truth (Quaker Midwife Mystery #1),20.89,2203.90,4,1
4,109,The Art of Startup Fundraising,21.00,2215.50,3,1
5,92,Voyager (Outlander #3),21.07,2222.89,5,1
6,110,Born for This: How to Find the Work You Were M...,21.59,2277.74,5,1
7,34,The Silkworm (Cormoran Strike #2),23.05,2431.78,5,1
8,9,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1,1
9,116,The Elephant Tree,23.82,2513.01,5,1


In [112]:
# Save Query 4 output

query4_output = os.path.join(
    OUTPUT_DIR,
    "query4_price_between_20_30.csv"
)

query4_result.to_csv(
    query4_output,
    index=False
)

print(f"Query 4 output saved to: {query4_output}")

Query 4 output saved to: outputs\query4_price_between_20_30.csv


In [113]:
# Validate Query 4

print("Number of books returned:", len(query4_result))

assert (
    (query4_result["price_gbp"] >= 20) &
    (query4_result["price_gbp"] <= 30)
).all(), "Some results fall outside the £20–£30 range."

print("✅ Query 4 validation passed.")

Number of books returned: 32
✅ Query 4 validation passed.


### Query 5 — Books in Selected Categories

Retrieve books belonging to the Travel, Mystery, or Science categories using the SQL IN operator.

In [114]:
# ==========================================
# SQL Query 5 — IN
# ==========================================

query5 = """
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
WHERE c.category_name IN ('Travel', 'Mystery', 'Science')
ORDER BY c.category_name, b.title
"""

query5_result = pd.read_sql(query5, conn)

print("Query 5 Results:")
query5_result

Query 5 Results:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,43,1st to Die (Women's Murder Club #1),53.98,5694.89,1,1,Mystery
1,15,A Murder in Time,16.64,1755.52,1,1,Mystery
2,21,A Study in Scarlet (Sherlock Holmes #1),16.73,1765.02,2,1,Mystery
3,20,A Time of Torment (Charlie Parker #14),48.35,5100.92,5,1,Mystery
4,42,Blood Defense (Samantha Brinkman #1),20.30,2141.65,3,1,Mystery
5,26,Boar Island (Anna Pigeon #19),59.48,6275.14,3,1,Mystery
6,38,Career of Evil (Cormoran Strike #3),24.72,2607.96,2,1,Mystery
7,31,Delivering the Truth (Quaker Midwife Mystery #1),20.89,2203.90,4,1,Mystery
8,37,Extreme Prey (Lucas Davenport #26),25.40,2679.70,3,1,Mystery
9,25,Hide Away (Eve Duncan #20),11.84,1249.12,1,1,Mystery


In [115]:
# Save Query 5 output

query5_output = os.path.join(
    OUTPUT_DIR,
    "query5_selected_categories.csv"
)

query5_result.to_csv(
    query5_output,
    index=False
)

print(f"Query 5 output saved to: {query5_output}")

Query 5 output saved to: outputs\query5_selected_categories.csv


In [116]:
# Validate Query 5

allowed_categories = {
    "Travel",
    "Mystery",
    "Science"
}

returned_categories = set(
    query5_result["category_name"]
)

print("Categories returned:", returned_categories)

assert returned_categories.issubset(
    allowed_categories
), "Unexpected category found."

assert len(query5_result) > 0, (
    "Query 5 returned no records."
)

print("✅ Query 5 validation passed.")

Categories returned: {'Travel', 'Science', 'Mystery'}
✅ Query 5 validation passed.


### Query 6 — Books with Category Information

Join the books and categories tables to retrieve book-level pricing, rating, availability, and category information.

This query will later be reproduced using pandas `pd.merge()` to verify that the SQL and pandas approaches produce equivalent results.

In [117]:
# ==========================================
# SQL Query 6 — JOIN
# ==========================================

query6 = """
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
INNER JOIN categories c
    ON b.category_id = c.category_id
ORDER BY c.category_name, b.title
LIMIT 20
"""

query6_result = pd.read_sql(query6, conn)

print("Query 6 Results:")
query6_result

Query 6 Results:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,110,Born for This: How to Find the Work You Were M...,21.59,2277.74,5,1,Business
1,107,Made to Stick: Why Some Ideas Survive and Othe...,38.85,4098.68,5,1,Business
2,108,Quench Your Own Thirst: Business Lessons Learn...,43.14,4551.27,1,1,Business
3,114,Rework,44.88,4734.84,2,1,Business
4,112,"Rich Dad, Poor Dad",51.74,5458.57,1,1,Business
5,106,Shoe Dog: A Memoir by the Creator of NIKE,23.99,2530.94,2,1,Business
6,105,The 10% Entrepreneur: Live Your Startup Dream ...,27.55,2906.52,3,1,Business
7,109,The Art of Startup Fundraising,21.00,2215.50,3,1,Business
8,103,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.37,4,1,Business
9,111,The E-Myth Revisited: Why Most Small Businesse...,36.91,3894.00,1,1,Business


In [118]:
# Save Query 6 output

query6_output = os.path.join(
    OUTPUT_DIR,
    "query6_join_books_categories.csv"
)

query6_result.to_csv(
    query6_output,
    index=False
)

print(f"Query 6 output saved to: {query6_output}")

Query 6 output saved to: outputs\query6_join_books_categories.csv


In [119]:
# ==========================================
# Validate Query 6
# ==========================================

print("Number of rows returned:", len(query6_result))

assert len(query6_result) == 20, (
    "Query 6 should return exactly 20 rows."
)

assert "category_name" in query6_result.columns, (
    "Category name is missing from JOIN result."
)

assert query6_result["category_name"].notna().all(), (
    "JOIN produced missing category names."
)

print("✅ Query 6 validation passed.")

Number of rows returned: 20
✅ Query 6 validation passed.


In [121]:
# ==========================================
# Read Database Tables into pandas
# ==========================================

books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

print("Books DataFrame shape:", books_df.shape)
print("Categories DataFrame shape:", categories_df.shape)

Books DataFrame shape: (125, 7)
Categories DataFrame shape: (7, 2)


In [122]:
# ==========================================
# Reproduce SQL JOIN using pandas
# ==========================================

pandas_join_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

pandas_join_result = pandas_join_result[
    [
        "book_id",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

pandas_join_result = pandas_join_result.sort_values(
    ["category_name", "title"]
).head(20).reset_index(drop=True)

print("Pandas JOIN result:")
pandas_join_result

Pandas JOIN result:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,110,Born for This: How to Find the Work You Were M...,21.59,2277.74,5,1,Business
1,107,Made to Stick: Why Some Ideas Survive and Othe...,38.85,4098.68,5,1,Business
2,108,Quench Your Own Thirst: Business Lessons Learn...,43.14,4551.27,1,1,Business
3,114,Rework,44.88,4734.84,2,1,Business
4,112,"Rich Dad, Poor Dad",51.74,5458.57,1,1,Business
5,106,Shoe Dog: A Memoir by the Creator of NIKE,23.99,2530.94,2,1,Business
6,105,The 10% Entrepreneur: Live Your Startup Dream ...,27.55,2906.52,3,1,Business
7,109,The Art of Startup Fundraising,21.00,2215.50,3,1,Business
8,103,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.37,4,1,Business
9,111,The E-Myth Revisited: Why Most Small Businesse...,36.91,3894.00,1,1,Business


In [123]:
# ==========================================
# Compare SQL JOIN vs pandas.merge()
# ==========================================

sql_result_comparison = query6_result.reset_index(drop=True)

pandas_result_comparison = pandas_join_result.reset_index(drop=True)

# Ensure identical column order
pandas_result_comparison = pandas_result_comparison[
    sql_result_comparison.columns
]

comparison = sql_result_comparison.equals(
    pandas_result_comparison
)

print("SQL JOIN and pandas.merge() equivalent:", comparison)

SQL JOIN and pandas.merge() equivalent: True


In [124]:
# ==========================================
# Side-by-Side Comparison
# ==========================================

comparison_df = pd.concat(
    [
        sql_result_comparison.add_prefix("SQL_"),
        pandas_result_comparison.add_prefix("PANDAS_")
    ],
    axis=1
)

comparison_df

,SQL_book_id,SQL_title,SQL_price_gbp,SQL_price_inr,SQL_rating,SQL_in_stock,SQL_category_name,PANDAS_book_id,PANDAS_title,PANDAS_price_gbp,PANDAS_price_inr,PANDAS_rating,PANDAS_in_stock,PANDAS_category_name
0,110,Born for This: How to Find the Work You Were M...,21.59,2277.74,5,1,Business,110,Born for This: How to Find the Work You Were M...,21.59,2277.74,5,1,Business
1,107,Made to Stick: Why Some Ideas Survive and Othe...,38.85,4098.68,5,1,Business,107,Made to Stick: Why Some Ideas Survive and Othe...,38.85,4098.68,5,1,Business
2,108,Quench Your Own Thirst: Business Lessons Learn...,43.14,4551.27,1,1,Business,108,Quench Your Own Thirst: Business Lessons Learn...,43.14,4551.27,1,1,Business
3,114,Rework,44.88,4734.84,2,1,Business,114,Rework,44.88,4734.84,2,1,Business
4,112,"Rich Dad, Poor Dad",51.74,5458.57,1,1,Business,112,"Rich Dad, Poor Dad",51.74,5458.57,1,1,Business
5,106,Shoe Dog: A Memoir by the Creator of NIKE,23.99,2530.94,2,1,Business,106,Shoe Dog: A Memoir by the Creator of NIKE,23.99,2530.94,2,1,Business
6,105,The 10% Entrepreneur: Live Your Startup Dream ...,27.55,2906.52,3,1,Business,105,The 10% Entrepreneur: Live Your Startup Dream ...,27.55,2906.52,3,1,Business
7,109,The Art of Startup Fundraising,21.00,2215.50,3,1,Business,109,The Art of Startup Fundraising,21.00,2215.50,3,1,Business
8,103,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.37,4,1,Business,103,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.37,4,1,Business
9,111,The E-Myth Revisited: Why Most Small Businesse...,36.91,3894.00,1,1,Business,111,The E-Myth Revisited: Why Most Small Businesse...,36.91,3894.00,1,1,Business


In [125]:
# ==========================================
# Final Equivalence Validation
# ==========================================

assert sql_result_comparison.equals(
    pandas_result_comparison
), "SQL and pandas JOIN results do not match."

print("✅ SQL JOIN and pandas.merge() produce equivalent results.")

✅ SQL JOIN and pandas.merge() produce equivalent results.


### SQL Query Log

The following queries were executed against the SQLite database and their outputs were saved under the `outputs/` directory.

In [126]:
# ==========================================
# Consolidated SQL Query Log
# ==========================================

sql_queries = {
    "Q1_SELECT_WHERE": query1,
    "Q2_ORDER_BY_LIMIT": query2,
    "Q3_DISTINCT": query3,
    "Q4_BETWEEN": query4,
    "Q5_IN": query5,
    "Q6_JOIN": query6
}

for query_name, query_string in sql_queries.items():
    print("=" * 70)
    print(query_name)
    print("=" * 70)
    print(query_string.strip())
    print()

Q1_SELECT_WHERE
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE price_gbp > 40

Q2_ORDER_BY_LIMIT
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
ORDER BY price_gbp DESC
LIMIT 10

Q3_DISTINCT
SELECT DISTINCT
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
ORDER BY c.category_name

Q4_BETWEEN
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock
FROM books
WHERE price_gbp BETWEEN 20 AND 30
ORDER BY price_gbp

Q5_IN
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
WHERE c.category_name IN ('Travel', 'Mystery', 'Science')
ORDER BY c.category_name, b.title

Q6_JOIN
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name

In [127]:
# ==========================================
# Save SQL Query Strings
# ==========================================

sql_log_file = os.path.join(
    OUTPUT_DIR,
    "sql_queries.txt"
)

with open(sql_log_file, "w", encoding="utf-8") as file:

    for query_name, query_string in sql_queries.items():

        file.write("=" * 70 + "\n")
        file.write(f"{query_name}\n")
        file.write("=" * 70 + "\n")
        file.write(query_string.strip() + "\n\n")

print(f"✅ SQL query log saved to: {sql_log_file}")

✅ SQL query log saved to: outputs\sql_queries.txt


In [128]:
# ==========================================
# Verify SQL Output Files
# ==========================================

expected_output_files = [
    "query1_price_above_40.csv",
    "query2_top10_expensive.csv",
    "query3_categories.csv",
    "query4_price_between_20_30.csv",
    "query5_selected_categories.csv",
    "query6_join_books_categories.csv",
    "sql_queries.txt"
]

for filename in expected_output_files:

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    print(
        f"{filename}:",
        "✅" if os.path.exists(filepath) else "❌"
    )

query1_price_above_40.csv: ✅
query2_top10_expensive.csv: ✅
query3_categories.csv: ✅
query4_price_between_20_30.csv: ✅
query5_selected_categories.csv: ✅
query6_join_books_categories.csv: ✅
sql_queries.txt: ✅


In [129]:
# ==========================================
# Stage 5 Final Validation
# ==========================================

assert len(query1_result) > 0
assert len(query2_result) == 10
assert query3_result["category_name"].is_unique
assert (
    (query4_result["price_gbp"] >= 20) &
    (query4_result["price_gbp"] <= 30)
).all()
assert set(query5_result["category_name"]).issubset(
    {"Travel", "Mystery", "Science"}
)
assert len(query6_result) == 20
assert sql_result_comparison.equals(
    pandas_result_comparison
)

print("==========================================")
print("✅ STAGE 5 SQL ANALYSIS PASSED")
print("==========================================")
print("Q1 SELECT + WHERE       ✅")
print("Q2 ORDER BY + LIMIT     ✅")
print("Q3 DISTINCT             ✅")
print("Q4 BETWEEN              ✅")
print("Q5 IN                   ✅")
print("Q6 JOIN                 ✅")
print("pd.read_sql             ✅")
print("pd.merge                ✅")
print("SQL/Pandas equivalence  ✅")
print("Query outputs saved     ✅")

✅ STAGE 5 SQL ANALYSIS PASSED
Q1 SELECT + WHERE       ✅
Q2 ORDER BY + LIMIT     ✅
Q3 DISTINCT             ✅
Q4 BETWEEN              ✅
Q5 IN                   ✅
Q6 JOIN                 ✅
pd.read_sql             ✅
pd.merge                ✅
SQL/Pandas equivalence  ✅
Query outputs saved     ✅


## 6. Pandas Analysis & Validation

The SQL query results are read into pandas DataFrames for further analysis.

At least two SQL results are loaded using `pd.read_sql()`. The SQL JOIN is independently reproduced using `pd.merge()` on in-memory DataFrames, and the results are compared for equivalence.

In [130]:
# ==========================================
# Read SQL Results into pandas
# ==========================================

pandas_q1 = pd.read_sql(query1, conn)
pandas_q2 = pd.read_sql(query2, conn)

print("Query 1 DataFrame shape:", pandas_q1.shape)
print("Query 2 DataFrame shape:", pandas_q2.shape)

print("\nQuery 1 result:")
display(pandas_q1.head())

print("\nQuery 2 result:")
display(pandas_q2.head())

Query 1 DataFrame shape: (41, 6)
Query 2 DataFrame shape: (10, 6)

Query 1 result:


,book_id,title,price_gbp,price_inr,rating,in_stock
0,1,It's Only the Himalayas,45.17,4765.44,2,1
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1
3,6,A Summer In Europe,44.34,4677.87,2,1
4,8,A Year in Provence (Provence #1),56.88,6000.84,4,1



Query 2 result:


,book_id,title,price_gbp,price_inr,rating,in_stock
0,123,The Bone Hunters (Lexy Vaughan & Steven Macaul...,59.71,6299.40,3,1
1,26,Boar Island (Anna Pigeon #19),59.48,6275.14,3,1
2,64,Candide,58.63,6185.46,3,1
3,39,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35,4,1
4,45,Immunity: How Elie Metchnikoff Changed the Cou...,57.36,6051.48,5,1


In [131]:
# ==========================================
# Validate pandas SQL Results
# ==========================================

assert isinstance(pandas_q1, pd.DataFrame)
assert isinstance(pandas_q2, pd.DataFrame)

assert (pandas_q1["price_gbp"] > 40).all()

assert len(pandas_q2) == 10

print("✅ Q1 successfully loaded into pandas.")
print("✅ Q2 successfully loaded into pandas.")
print("✅ Both pd.read_sql() requirements satisfied.")

✅ Q1 successfully loaded into pandas.
✅ Q2 successfully loaded into pandas.
✅ Both pd.read_sql() requirements satisfied.


In [132]:
# ==========================================
# In-Memory DataFrames
# ==========================================

print("Books DataFrame:")
display(books_df.head())

print("\nCategories DataFrame:")
display(categories_df.head())

Books DataFrame:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.44,2,1,1
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1,1
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,1,1
4,5,Under the Tuscan Sun,37.33,3938.31,3,1,1



Categories DataFrame:


,category_id,category_name
0,1,Travel
1,2,Mystery
2,3,Science
3,4,Classics
4,5,Historical Fiction


In [133]:
# ==========================================
# Pandas JOIN using pd.merge()
# ==========================================

pandas_join_full = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

pandas_join_full = pandas_join_full[
    [
        "book_id",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

pandas_join_full = pandas_join_full.sort_values(
    ["category_name", "title"]
).reset_index(drop=True)

print("Pandas JOIN shape:", pandas_join_full.shape)

display(pandas_join_full.head(20))

Pandas JOIN shape: (125, 7)


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,110,Born for This: How to Find the Work You Were M...,21.59,2277.74,5,1,Business
1,107,Made to Stick: Why Some Ideas Survive and Othe...,38.85,4098.68,5,1,Business
2,108,Quench Your Own Thirst: Business Lessons Learn...,43.14,4551.27,1,1,Business
3,114,Rework,44.88,4734.84,2,1,Business
4,112,"Rich Dad, Poor Dad",51.74,5458.57,1,1,Business
5,106,Shoe Dog: A Memoir by the Creator of NIKE,23.99,2530.94,2,1,Business
6,105,The 10% Entrepreneur: Live Your Startup Dream ...,27.55,2906.52,3,1,Business
7,109,The Art of Startup Fundraising,21.00,2215.50,3,1,Business
8,103,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.37,4,1,Business
9,111,The E-Myth Revisited: Why Most Small Businesse...,36.91,3894.00,1,1,Business


In [134]:
# ==========================================
# Match SQL Q6 Result
# ==========================================

pandas_join_20 = (
    pandas_join_full
    .head(20)
    .reset_index(drop=True)
)

sql_join_20 = (
    query6_result
    .reset_index(drop=True)
)

print("SQL JOIN rows:", len(sql_join_20))
print("Pandas JOIN rows:", len(pandas_join_20))

SQL JOIN rows: 20
Pandas JOIN rows: 20


In [135]:
# ==========================================
# SQL JOIN vs pandas.merge()
# ==========================================

pandas_join_20 = pandas_join_20[
    sql_join_20.columns
]

join_results_match = sql_join_20.equals(
    pandas_join_20
)

print(
    "SQL JOIN and pandas.merge() results match:",
    join_results_match
)

SQL JOIN and pandas.merge() results match: True


In [136]:
# ==========================================
# Final JOIN Equivalence Check
# ==========================================

assert join_results_match, (
    "SQL JOIN and pandas.merge() results do not match."
)

print(
    "✅ SQL JOIN and pandas.merge() "
    "produce equivalent results."
)

✅ SQL JOIN and pandas.merge() produce equivalent results.


In [137]:
# ==========================================
# Save Pandas JOIN Result
# ==========================================

pandas_join_output = os.path.join(
    OUTPUT_DIR,
    "pandas_merge_join_result.csv"
)

pandas_join_20.to_csv(
    pandas_join_output,
    index=False
)

print(
    f"✅ Pandas JOIN result saved to: "
    f"{pandas_join_output}"
)

✅ Pandas JOIN result saved to: outputs\pandas_merge_join_result.csv


In [138]:
# ==========================================
# Side-by-Side SQL vs Pandas Comparison
# ==========================================

comparison_display = pd.DataFrame({
    "SQL_book_id": sql_join_20["book_id"],
    "Pandas_book_id": pandas_join_20["book_id"],
    "SQL_category": sql_join_20["category_name"],
    "Pandas_category": pandas_join_20["category_name"],
    "SQL_price_gbp": sql_join_20["price_gbp"],
    "Pandas_price_gbp": pandas_join_20["price_gbp"]
})

display(comparison_display)

,SQL_book_id,Pandas_book_id,SQL_category,Pandas_category,SQL_price_gbp,Pandas_price_gbp
0,110,110,Business,Business,21.59,21.59
1,107,107,Business,Business,38.85,38.85
2,108,108,Business,Business,43.14,43.14
3,114,114,Business,Business,44.88,44.88
4,112,112,Business,Business,51.74,51.74
5,106,106,Business,Business,23.99,23.99
6,105,105,Business,Business,27.55,27.55
7,109,109,Business,Business,21.00,21.00
8,103,103,Business,Business,33.34,33.34
9,111,111,Business,Business,36.91,36.91


In [139]:
# ==========================================
# Stage 6 Final Validation
# ==========================================

assert isinstance(pandas_q1, pd.DataFrame)
assert isinstance(pandas_q2, pd.DataFrame)
assert len(pandas_q2) == 10
assert join_results_match

print("==========================================")
print("✅ STAGE 6 PANDAS ANALYSIS PASSED")
print("==========================================")
print("pd.read_sql() Query 1       ✅")
print("pd.read_sql() Query 2       ✅")
print("pd.merge() JOIN             ✅")
print("SQL/Pandas equivalence      ✅")
print("Comparison output displayed ✅")

✅ STAGE 6 PANDAS ANALYSIS PASSED
pd.read_sql() Query 1       ✅
pd.read_sql() Query 2       ✅
pd.merge() JOIN             ✅
SQL/Pandas equivalence      ✅
Comparison output displayed ✅


In [141]:
# ==========================================
# Repository Structure Validation
# ==========================================

required_files = [
    "data/raw_books.csv",
    "data/cleaned_books.csv",
    "data/books.db",
    "outputs/query1_price_above_40.csv",
    "outputs/query2_top10_expensive.csv",
    "outputs/query3_categories.csv",
    "outputs/query4_price_between_20_30.csv",
    "outputs/query5_selected_categories.csv",
    "outputs/query6_join_books_categories.csv",
    "outputs/pandas_merge_join_result.csv",
    "outputs/sql_queries.txt",
    "requirements.txt",
    "README.md"
]

print("Repository file validation:\n")

missing_files = []

for file_path in required_files:
    exists = os.path.exists(file_path)

    print(
        f"{'✅' if exists else '❌'} {file_path}"
    )

    if not exists:
        missing_files.append(file_path)

if missing_files:
    print("\nMissing files:")
    for file_path in missing_files:
        print("-", file_path)
else:
    print("\n✅ All required submission files exist.")

Repository file validation:

✅ data/raw_books.csv
✅ data/cleaned_books.csv
✅ data/books.db
✅ outputs/query1_price_above_40.csv
✅ outputs/query2_top10_expensive.csv
✅ outputs/query3_categories.csv
✅ outputs/query4_price_between_20_30.csv
✅ outputs/query5_selected_categories.csv
✅ outputs/query6_join_books_categories.csv
✅ outputs/pandas_merge_join_result.csv
✅ outputs/sql_queries.txt
✅ requirements.txt
✅ README.md

✅ All required submission files exist.
